In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_party_identification_qca(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "PartyIdentificationId string,"
        #     "SourceSystem string,"
        #     "PortfolioKey string,"
        #     "PortfolioName string,"
        #     "IngestionTimestamp timestamp,"
        #     "BatchId string,"
        #     "PartyIdentificationKey string,"
        #     "PartyKey string,"
        #     "CDI string,"
        #     "PartyIdentificationTypeCode string,"
        #     "PartyIdentificationType string,"
        #     "PartyIdentificationNumber string,"
        #     "SourceTable string,"
        #     "MainFlag string,"
        #     "IssueAuthority string,"
        #     "IssueDate date,"
        #     "ExpiredDate date,"
        #     "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.PartyIdentification"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "qca",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== dim_customer_static ==================

        dim_customer_static_extra_filters = None

        if dim_customer_static_extra_filters is not None:
            dim_customer_static_extra_filters.append(date_filter)
            dim_customer_static_final_filters = " AND ".join(
                dim_customer_static_extra_filters
            )
        else:
            dim_customer_static_final_filters = date_filter

        dim_customer_static_selected_fields = [
            "CUST_ID",
            "CUST_ID_TYPE",
            "CUST_ID_NUM",
            "cust_cif",
            "businessdate",
        ]

        dim_customer_static = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "qca_dim_customer_static",
            dim_customer_static_final_filters,
            dim_customer_static_selected_fields,
        )
        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter

        portfolio_selected_fields = [
            "prefix",
            "PortfolioName",
            "portfolio_id_qca",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )
        # ================== Join ==================
        PartyIdentificationId_primary_key_fields = [
            dim_customer_static.CUST_ID,
            dim_customer_static.CUST_ID_NUM,
            dim_customer_static.CUST_ID_TYPE,
            lit("qca_dim_customer_static"),
            lit("qca"),
        ]

        PartyIdentificationKey_primary_key_fields = [
            dim_customer_static.CUST_ID,
            dim_customer_static.CUST_ID_NUM,
            dim_customer_static.CUST_ID_TYPE,
            lit("qca_dim_customer_static"),
        ]
        party_identification_qca = dim_customer_static.join(
            portfolio,
            (substring(dim_customer_static["cust_cif"], 1, 3) == portfolio["prefix"]),
            "left",
        ).where(
            dim_customer_static.CUST_ID_NUM.isNotNull()
            & (dim_customer_static.CUST_ID_NUM != "")
        )
        # ================== party_identification_qca ==================
        party_identification_qca = party_identification_qca.withColumn(
            "PartyIdentificationId",
            sha2(concat_ws(";", *PartyIdentificationId_primary_key_fields), 256),
        ).withColumn(
            "PartyIdentificationKey",
            concat_ws(";", *PartyIdentificationKey_primary_key_fields),
        )

        party_identification_qca = party_identification_qca.select(
            party_identification_qca.PartyIdentificationId.cast("string").alias(
                "PartyIdentificationId"
            ),
            lit("qca").alias("SourceSystem"),
            coalesce(portfolio.portfolio_id_qca, lit(2))
            .cast("string")
            .alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Retail"))
            .cast("string")
            .alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            party_identification_qca.PartyIdentificationKey.cast("string").alias(
                "PartyIdentificationKey"
            ),
            party_identification_qca.CUST_ID.cast("string").alias("PartyKey"),
            party_identification_qca.cust_cif.cast("string").alias("CDI"),
            party_identification_qca.CUST_ID_TYPE.alias("PartyIdentificationTypeCode"),
            party_identification_qca.CUST_ID_TYPE.alias("PartyIdentificationType"),
            party_identification_qca.CUST_ID_NUM.cast("string").alias(
                "PartyIdentificationNumber"
            ),
            lit("qca_dim_customer_static").cast("string").alias("SourceTable"),
            lit(None).cast("string").alias("MainFlag"),
            lit(None).cast("string").alias("IssueAuthority"),
            lit(None).cast("date").alias("IssueDate"),
            lit(None).alias("ExpiredDate"),
            party_identification_qca.businessdate.alias("BusinessDate"),
        )
        
        # ================== Append ==================

        party_identification_qca.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")

In [0]:
load_curated_party_identification_qca(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)